In [52]:
import torch
import numpy as np
import scipy.special as sp

import pickle as pkl
import zlib
import base64

In [53]:
from src.envs.agents.dqn_agent_ext import DQNAgentExt
from src.game.template import calculate_output_np

In [54]:
import sys
sys.path.append('/Users/aleksei/projects/code-of-kutulu-client')

In [55]:
checkpoint_dir = '../output/2025-05-07/18:38:17.436318/agent0'

In [56]:
info = {
    'train': True,
    'qdn_ext': True,
    'state_type': 'closest_ext',
    'gamma': 0.9,
    'replay_size': 10000,
    'replay_start_size': 100,
    'sync_target_frames': 1000,
    'batch_size': 64,
}

In [57]:
del info['qdn_ext']
info['action_space_n'] = 5

In [58]:
agent = DQNAgentExt(**info)

In [59]:
agent.model.load_state_dict(torch.load(f"../output/2025-05-07/18:38:17.436318/agent0/model.pt"))
agent.tgt_net.load_state_dict(torch.load(f"../output/2025-05-07/18:38:17.436318/agent0/tgt_net.pt"))

<All keys matched successfully>

In [60]:
data = {'entity_kind': [[1, 1, 3, 2, 0, 0, 0, 0, 0, 0]],
 'entity_features': [[[218.0, 3.0, 3.0, -3.0, 6.0, 0.0, 218.0],
   [221.0, 3.0, 1.0, -9.0, 10.0, 0.0, 221.0],
   [27.0, 0.0, 6.0, 6.0, 12.0, 0.0, 27.0],
   [2.0, -1.0, 7.0, 4.0, 11.0, 0.0, 2.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]],
 'entity_dir': [[[0.0, 6, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 12, 0.0],
   [0.0, 12, 0.0, 0.0, 0.0],
   [0.0, 15, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0]]]}

In [61]:
weights = {}
for k,v in agent.model.named_parameters():
    weights[k] = v.detach().numpy()
    print(k, v.shape)

kind_embs.weight torch.Size([13, 32])
features_linear.weight torch.Size([32, 7])
features_linear.bias torch.Size([32])
dir_linear.weight torch.Size([16, 5])
dir_linear.bias torch.Size([16])
entity_linear.weight torch.Size([16, 80])
entity_linear.bias torch.Size([16])
entity_impact.weight torch.Size([5, 80])
entity_impact.bias torch.Size([5])
out_linear.weight torch.Size([1, 16])
out_linear.bias torch.Size([1])


In [62]:
calculate_output_np(data, weights)

array([[0.15718925, 0.16322085, 0.14833277, 0.15050976, 0.16069223]])

In [63]:
tensor_data = {k: torch.tensor(v) for k,v in data.items()}

In [64]:
tensor_data = {
    'entity_kind': torch.IntTensor(data['entity_kind']),
    'entity_features': torch.FloatTensor(data['entity_features']),
    'entity_dir': torch.FloatTensor(data['entity_dir']),
}

In [65]:
model_output = agent.model(tensor_data)[0].detach().cpu().numpy()

In [66]:
model_output

array([0.15718912, 0.16322069, 0.14833279, 0.15050973, 0.16069214],
      dtype=float32)

In [83]:
data2, data1 = zip(*weights.items())

data1 = pkl.dumps(data1)
data2 = pkl.dumps(data2)

In [84]:
with open('../src/game/template.py') as f:
    lines = f.readlines()

In [85]:
with open('../src/game/template_submit.py', 'w') as f:
    for line in lines:
        line = line.replace("b'data1data1data1'", str(base64.b64encode(zlib.compress(data1, level=9))))
        line = line.replace("b'data2data2data2'", str(base64.b64encode(zlib.compress(data2, level=9))))
        line = line.replace("mode = 'mode'", "mode = 'dqn_ext'")
        f.write(line)

In [86]:
!ls -lh ../src/game/template_submit.py

-rw-r--r--  1 aleksei  staff    27K  7 май 22:01 ../src/game/template_submit.py
